# 11. Modeling v2 — Train / Validation / Test dengan Macro F1

Implementasi `MODELING_STRATEGY.md` (eksperimen E1–E5).

**Struktur data (tanpa kebocoran):**

| Bagian | Isi | Dipakai untuk |
|---|---|---|
| **Train (dev)** | 85% data train berlabel, dibagi 5 fold | melatih model per fold |
| **Validation** | fold ke-k dari dev (bergiliran) | early stopping, OOF, tuning multiplier kelas |
| **Test (holdout)** | 15% data train berlabel, *stratified*, **tidak pernah dipakai** untuk fitting, TE, maupun tuning | **Macro F1 final yang jujur** |
| Test kompetisi | `test/` (tanpa label) | submission |

**Metrik evaluasi utama: Macro F1** (`f1_score(average="macro")`) di semua tahap.

Fitur:
- Skenario C + `iv_n_klaim`, `iv_amount_klaim`, `sc_n_selesai` (tersedia di test, bukan leak deterministik).
- Fitur redundan dibuang; `customer_id`/`courier_id` mentah diganti target encoding.
- Target encoding (TE) entitas customer/courier/hub **dengan smoothing**, dihitung ulang di sini:
  OOF di dalam dev, dan dari seluruh dev untuk holdout & test.
- Frekuensi entitas dan fitur deviasi (tanpa target, dihitung dari train+test).

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display

from analysis_data import CACHE_DIR, DATA_DIR, load_analysis_frame

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 5
HOLDOUT_SIZE = 0.15
TARGET = "status_penanganan"
ID_COL = "shipment_id"

ENTITY_COLUMNS = ["customer_id", "courier_id", "hub_id"]
TE_SMOOTHING = {"customer_id": 10, "courier_id": 20, "hub_id": 50}

CLASS_WEIGHT_MODE = "sqrt"        # None | "sqrt" | "balanced"
USE_CATBOOST = False              # opsional, lambat di CPU
RUN_ABLATION = False              # opsional, menjalankan beberapa CV tambahan
FINAL_REFIT_FULL_TRAIN = False    # True: setelah evaluasi, latih ulang di 100% train untuk submission

OUTPUT_DIR = Path("section11_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 11.1 Load data 

In [ ]:
train_path = CACHE_DIR / "analysis_frame_train.parquet"
test_path = CACHE_DIR / "analysis_frame_test.parquet"

for split, path in [("train", train_path), ("test", test_path)]:
    if not path.exists():
        _tmp = load_analysis_frame(split)
        del _tmp
        gc.collect()

train_names = pl.scan_parquet(train_path).collect_schema().names()
test_names = set(pl.scan_parquet(test_path).collect_schema().names())
common_columns = [c for c in train_names if c in test_names]

# Redundan (section 06): pasangan identik / hampir identik -> simpan satu
REDUNDANT_FEATURES = [
    "rt_distance_sum",        # = distance_km
    "x_address_risk",         # = x_address_notes
    "iv_count",               # ~ iv_n_klaim
    "iv_penalty_count",       # = iv_n_gagal
    "iv_discount_missing",    # = iv_n_langganan
    "iv_tax_sum",             # ~ iv_amount_sum
    "sc_unique_hubs",         # = rt_unique_hubs
    "promo_nama_promo",       # <-> promo_id
    "promo_tipe_promo",       # <-> promo_id
    "pkg_speed_class",        # <-> package_id
    "pkg_handling_class",     # <-> package_id
]

# ID mentah berkardinalitas tinggi diwakili target encoding
DROP_COLUMNS = {ID_COL, TARGET, "customer_id", "courier_id"}

feature_columns = [
    c for c in common_columns
    if c not in DROP_COLUMNS and c not in REDUNDANT_FEATURES
]

load_columns = list(dict.fromkeys([ID_COL, TARGET] + ENTITY_COLUMNS + feature_columns))

train_pl = pl.read_parquet(train_path, columns=load_columns)
test_pl = pl.read_parquet(test_path, columns=[c for c in load_columns if c != TARGET])

if "cour_hub_id" in feature_columns and (
    train_pl["cour_hub_id"].cast(pl.Utf8) == train_pl["hub_id"].cast(pl.Utf8)
).all():
    feature_columns.remove("cour_hub_id")
    print("cour_hub_id identik dengan hub_id -> dibuang")

# String -> Enum dengan kategori gabungan train+test (kode konsisten, null -> __MISSING__)
string_columns = [
    c for c, d in train_pl.schema.items()
    if c not in (ID_COL, TARGET) and (d == pl.String or d == pl.Categorical)
]

for col in string_columns:
    tr = train_pl[col].cast(pl.Utf8).fill_null("__MISSING__")
    te = test_pl[col].cast(pl.Utf8).fill_null("__MISSING__")
    enum = pl.Enum(pl.concat([tr, te]).unique().sort())
    train_pl = train_pl.with_columns(tr.cast(enum).alias(col))
    test_pl = test_pl.with_columns(te.cast(enum).alias(col))

numeric_casts = [
    pl.col(c).cast(pl.Float32)
    for c, d in train_pl.schema.items()
    if c not in (ID_COL, TARGET) and c not in string_columns and d.is_numeric()
]
train_pl = train_pl.with_columns(numeric_casts)
test_pl = test_pl.with_columns(numeric_casts)

# Kode entitas (int32) untuk TE / frekuensi
ent_codes_train, ent_codes_test, ent_n_keys = {}, {}, {}
for e in ENTITY_COLUMNS:
    ent_codes_train[e] = train_pl[e].to_physical().to_numpy().astype(np.int64)
    ent_codes_test[e] = test_pl[e].to_physical().to_numpy().astype(np.int64)
    ent_n_keys[e] = len(train_pl[e].dtype.categories)

y_labels = train_pl[TARGET].cast(pl.Utf8).to_numpy()
class_labels = sorted(pd.unique(y_labels).tolist())
K = len(class_labels)
y_int = pd.Categorical(y_labels, categories=class_labels).codes.astype(np.int64)

test_ids = test_pl[ID_COL].to_numpy()

train_df = train_pl.select(feature_columns).to_pandas()
test_df = test_pl.select(feature_columns).to_pandas()

n_train, n_test = len(train_df), len(test_df)

del train_pl, test_pl
gc.collect()

print(f"Train: {train_df.shape} | Test kompetisi: {test_df.shape}")
print(f"Kelas ({K}): {class_labels}")
print(f"Fitur basis: {len(feature_columns)} | kategorikal: "
      f"{sum(isinstance(train_df[c].dtype, pd.CategoricalDtype) for c in feature_columns)}")
print(f"RAM train_df: {train_df.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

## 11.2 Split Train / Validation / Test (holdout)

In [ ]:
all_idx = np.arange(n_train)

dev_idx, holdout_idx = train_test_split(
    all_idx,
    test_size=HOLDOUT_SIZE,
    stratify=y_int,
    random_state=RANDOM_STATE,
)
dev_idx = np.sort(dev_idx)
holdout_idx = np.sort(holdout_idx)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

fold_of = np.full(n_train, -1, dtype=np.int8)   # -1 = holdout
for k, (_, va) in enumerate(skf.split(dev_idx, y_int[dev_idx])):
    fold_of[dev_idx[va]] = k

dev_folds = fold_of[dev_idx]

assert len(np.intersect1d(dev_idx, holdout_idx)) == 0
assert (fold_of[holdout_idx] == -1).all()

split_table = pd.DataFrame({
    "train_dev (5-fold)": np.bincount(y_int[dev_idx], minlength=K),
    "test_holdout": np.bincount(y_int[holdout_idx], minlength=K),
}, index=class_labels)
split_table.loc["TOTAL"] = split_table.sum()
split_pct = (split_table.iloc[:-1] / split_table.iloc[:-1].sum() * 100).round(2)

print(f"Train (dev)   : {len(dev_idx):,} baris -> per fold train ~{len(dev_idx) * 4 // 5:,} / validation ~{len(dev_idx) // 5:,}")
print(f"Test (holdout): {len(holdout_idx):,} baris")
print(f"Test kompetisi: {n_test:,} baris")
display(split_table)
display(split_pct.add_suffix(" %"))

## 11.3 Target encoding entitas (smoothed, tanpa leakage)

`rate = (n_kelas + m * prior) / (n + m)`

- Baris **dev**: OOF, sehingga fold *k* di-encode dari dev fold lain.
- Baris **holdout** dan **test kompetisi**: di-encode dari seluruh dev. Label holdout tidak pernah dipakai.

In [ ]:
def target_encode(ref_codes, ref_y, query_codes, n_keys, m):
    prior = np.bincount(ref_y, minlength=K) / len(ref_y)
    counts = np.bincount(ref_codes, minlength=n_keys).astype(np.float64)
    class_counts = np.zeros((n_keys, K), dtype=np.float64)
    for k in range(K):
        class_counts[:, k] = np.bincount(ref_codes[ref_y == k], minlength=n_keys)
    rates = (class_counts + m * prior) / (counts[:, None] + m)
    return rates[query_codes], counts[query_codes]


def pack_te(rates, counts):
    entropy = -(rates * np.log(np.clip(rates, 1e-12, None))).sum(axis=1)
    return np.column_stack([
        rates,
        entropy,
        rates.max(axis=1),
        np.log1p(counts),
        (counts == 0).astype(np.float64),
    ]).astype(np.float32)


def te_feature_names(entity):
    name = entity.replace("_id", "")
    return (
        [f"te_{name}_rate_{label}" for label in class_labels]
        + [f"te_{name}_entropy", f"te_{name}_max_rate",
           f"te_{name}_log_count", f"te_{name}_is_unseen"]
    )


def attach_target_encoding(fit_idx, fit_folds, extra_idx):
    # Tulis kolom TE ke train_df / test_df (in-place).
    # fit_idx   : baris train untuk fitting (di-encode OOF per fold)
    # extra_idx : baris train lain (holdout) di-encode dari seluruh fit_idx
    names_all = []
    for e in ENTITY_COLUMNS:
        m = TE_SMOOTHING[e]
        ctr, cte, nk = ent_codes_train[e], ent_codes_test[e], ent_n_keys[e]
        names = te_feature_names(e)
        block = np.full((n_train, len(names)), np.nan, dtype=np.float32)

        for k in range(N_SPLITS):
            ref = fit_idx[fit_folds != k]
            qry = fit_idx[fit_folds == k]
            block[qry] = pack_te(*target_encode(ctr[ref], y_int[ref], ctr[qry], nk, m))

        if len(extra_idx):
            block[extra_idx] = pack_te(
                *target_encode(ctr[fit_idx], y_int[fit_idx], ctr[extra_idx], nk, m)
            )

        block_test = pack_te(*target_encode(ctr[fit_idx], y_int[fit_idx], cte, nk, m))

        for j, col in enumerate(names):
            train_df[col] = block[:, j]
            test_df[col] = block_test[:, j]

        names_all += names
        del block, block_test
        gc.collect()

    return names_all


TE_FEATURES = attach_target_encoding(dev_idx, dev_folds, holdout_idx)

assert not train_df[TE_FEATURES].isna().any().any(), "Ada baris train tanpa TE"
print(f"Fitur TE: {len(TE_FEATURES)}")
display(train_df[TE_FEATURES].describe().T.head(12))

## 11.4 Frekuensi entitas, fitur deviasi, dan indikator missing (tanpa target)

In [ ]:
FREQ_DEV_FEATURES = []


def entity_all_codes(e):
    return np.concatenate([ent_codes_train[e], ent_codes_test[e]])


def entity_mean(e, col):
    codes = entity_all_codes(e)
    vals = np.concatenate([
        train_df[col].to_numpy(dtype=np.float64),
        test_df[col].to_numpy(dtype=np.float64),
    ])
    valid = ~np.isnan(vals)
    sums = np.bincount(codes[valid], weights=vals[valid], minlength=ent_n_keys[e])
    n = np.bincount(codes[valid], minlength=ent_n_keys[e])
    with np.errstate(invalid="ignore", divide="ignore"):
        means = sums / n
    return means[ent_codes_train[e]], means[ent_codes_test[e]]


def add_feature(name, train_values, test_values):
    train_df[name] = np.asarray(train_values, dtype=np.float32)
    test_df[name] = np.asarray(test_values, dtype=np.float32)
    FREQ_DEV_FEATURES.append(name)


# Frekuensi shipment per entitas di train+test
for e in ENTITY_COLUMNS:
    counts = np.bincount(entity_all_codes(e), minlength=ent_n_keys[e])
    add_feature(f"freq_{e.replace('_id', '')}", counts[ent_codes_train[e]], counts[ent_codes_test[e]])

# Deviasi shipment terhadap profil entitasnya
deviation_specs = [
    ("dev_value_vs_customer", "customer_id", "declared_value", "ratio"),
    ("dev_weight_vs_customer", "customer_id", "weight_kg", "ratio"),
    ("dev_distance_vs_courier", "courier_id", "distance_km", "ratio"),
    ("dev_value_vs_courier", "courier_id", "declared_value", "ratio"),
    ("dev_courier_score_vs_hub", "hub_id", "cour_courier_score", "diff"),
    ("dev_address_quality_vs_hub", "hub_id", "cust_address_quality", "diff"),
]

for name, e, col, kind in deviation_specs:
    if col not in train_df.columns:
        continue
    mean_tr, mean_te = entity_mean(e, col)
    val_tr = train_df[col].to_numpy(dtype=np.float64)
    val_te = test_df[col].to_numpy(dtype=np.float64)
    with np.errstate(invalid="ignore", divide="ignore"):
        if kind == "ratio":
            f_tr = np.where(mean_tr > 0, val_tr / mean_tr, np.nan)
            f_te = np.where(mean_te > 0, val_te / mean_te, np.nan)
        else:
            f_tr, f_te = val_tr - mean_tr, val_te - mean_te
    add_feature(name, f_tr, f_te)

# Indikator missing
for col in ["cust_credit_score", "cour_courier_score", "volume_l"]:
    if col in train_df.columns:
        add_feature(f"{col}_isna", train_df[col].isna(), test_df[col].isna())

for frame in (train_df, test_df):
    for col in frame.columns:
        if frame[col].dtype.kind == "f":
            values = frame[col].to_numpy()
            if np.isinf(values).any():
                frame[col] = np.where(np.isinf(values), np.nan, values).astype(np.float32)

BASE_FEATURES = list(feature_columns)
ALL_FEATURES = BASE_FEATURES + TE_FEATURES + FREQ_DEV_FEATURES

assert list(train_df.columns) == list(test_df.columns)
print(f"Fitur basis: {len(BASE_FEATURES)} | TE: {len(TE_FEATURES)} | freq/deviasi: {len(FREQ_DEV_FEATURES)}")
print(f"Total fitur: {len(ALL_FEATURES)}")

## 11.5 Metrik Macro F1, bobot kelas, dan fungsi CV

In [ ]:
def macro_f1(y_true, y_pred):
    # Macro F1 cepat via confusion matrix (identik dengan f1_score(average="macro"))
    cm = np.bincount(y_true * K + y_pred, minlength=K * K).reshape(K, K)
    tp = np.diag(cm).astype(np.float64)
    denom = 2 * tp + (cm.sum(0) - tp) + (cm.sum(1) - tp)
    return float(np.mean(np.where(denom > 0, 2 * tp / denom, 0.0)))


def make_class_weights(y_ref, mode):
    counts = np.bincount(y_ref, minlength=K).astype(np.float64)
    balanced = len(y_ref) / (K * counts)
    if mode is None:
        return np.ones(K)
    if mode == "sqrt":
        return np.sqrt(balanced)
    if mode == "balanced":
        return balanced
    raise ValueError(mode)


lgb_params = {
    "objective": "multiclass",
    "learning_rate": 0.03,
    "n_estimators": 5000,
    "num_leaves": 127,
    "min_child_samples": 200,
    "colsample_bytree": 0.6,
    "subsample": 0.8,
    "subsample_freq": 1,
    "reg_alpha": 0.5,
    "reg_lambda": 2.0,
    "cat_smooth": 50,
    "min_data_per_group": 200,
    "max_cat_to_onehot": 8,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
}

class_weights = make_class_weights(y_int[dev_idx], CLASS_WEIGHT_MODE)
display(pd.DataFrame({"class": class_labels, "weight": class_weights.round(3)}))


def run_lgb_cv(features, fit_idx, fit_folds, eval_idx, params,
               weight_mode=CLASS_WEIGHT_MODE, log_period=200):
    # CV LightGBM: train/validation per fold. Return OOF, prediksi holdout & test (rata-rata 5 fold).
    # Hindari copy penuh jika memakai semua kolom (hemat RAM)
    if list(features) == list(train_df.columns):
        X, X_test_ = train_df, test_df
    else:
        X, X_test_ = train_df[features], test_df[features]
    w_class = make_class_weights(y_int[fit_idx], weight_mode)

    oof = np.zeros((n_train, K), dtype=np.float32)
    eval_pred = np.zeros((len(eval_idx), K), dtype=np.float64)
    test_pred = np.zeros((n_test, K), dtype=np.float64)
    fold_rows, importances = [], []

    for k in range(N_SPLITS):
        tr = fit_idx[fit_folds != k]
        va = fit_idx[fit_folds == k]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X.iloc[tr], y_int[tr],
            sample_weight=w_class[y_int[tr]],
            eval_set=[(X.iloc[va], y_int[va])],
            eval_metric="multi_logloss",
            callbacks=[
                lgb.early_stopping(stopping_rounds=200, verbose=False),
                lgb.log_evaluation(period=log_period),
            ],
        )

        best_it = model.best_iteration_
        oof[va] = model.predict_proba(X.iloc[va], num_iteration=best_it)
        if len(eval_idx):
            eval_pred += model.predict_proba(X.iloc[eval_idx], num_iteration=best_it) / N_SPLITS
        test_pred += model.predict_proba(X_test_, num_iteration=best_it) / N_SPLITS

        score = macro_f1(y_int[va], oof[va].argmax(1))
        fold_rows.append({"fold": k + 1, "val_macro_f1": score, "best_iteration": best_it})
        importances.append(pd.DataFrame({
            "feature": features,
            "gain": model.booster_.feature_importance(importance_type="gain"),
            "fold": k + 1,
        }))
        print(f"Fold {k + 1}: validation Macro F1 = {score:.5f} | best_iteration = {best_it}")

        del model
        gc.collect()

    del X, X_test_
    gc.collect()

    return {
        "oof": oof,
        "eval_pred": eval_pred.astype(np.float32),
        "test_pred": test_pred.astype(np.float32),
        "fold_scores": pd.DataFrame(fold_rows),
        "importance": pd.concat(importances, ignore_index=True),
    }

## 11.6 Training LightGBM (5-fold pada Train, validasi per fold)

In [ ]:
lgb_result = run_lgb_cv(ALL_FEATURES, dev_idx, dev_folds, holdout_idx, lgb_params)

fold_scores = lgb_result["fold_scores"]
display(fold_scores)

dev_oof_f1 = macro_f1(y_int[dev_idx], lgb_result["oof"][dev_idx].argmax(1))
holdout_f1 = macro_f1(y_int[holdout_idx], lgb_result["eval_pred"].argmax(1))

print(f"Validation Macro F1 (mean ± std fold): {fold_scores['val_macro_f1'].mean():.5f} ± {fold_scores['val_macro_f1'].std():.5f}")
print(f"Validation Macro F1 (OOF dev)        : {dev_oof_f1:.5f}")
print(f"Test holdout Macro F1 (argmax)       : {holdout_f1:.5f}")

final_dev_proba = lgb_result["oof"][dev_idx]
final_holdout_proba = lgb_result["eval_pred"]
final_test_proba = lgb_result["test_pred"]
final_model_name = "LightGBM"

## 11.7 (Opsional) CatBoost + ensemble

In [ ]:
if USE_CATBOOST:
    from catboost import CatBoostClassifier, Pool

    cat_features = [c for c in ALL_FEATURES if isinstance(train_df[c].dtype, pd.CategoricalDtype)]
    w_class = make_class_weights(y_int[dev_idx], CLASS_WEIGHT_MODE)

    cb_oof = np.zeros((n_train, K), dtype=np.float32)
    cb_holdout = np.zeros((len(holdout_idx), K), dtype=np.float64)
    cb_test = np.zeros((n_test, K), dtype=np.float64)

    for k in range(N_SPLITS):
        tr = dev_idx[dev_folds != k]
        va = dev_idx[dev_folds == k]
        model = CatBoostClassifier(
            loss_function="MultiClass", depth=8, learning_rate=0.08, iterations=3000,
            od_type="Iter", od_wait=200, random_seed=RANDOM_STATE, thread_count=-1,
            verbose=200,
        )
        model.fit(
            Pool(train_df.iloc[tr][ALL_FEATURES], y_int[tr], cat_features=cat_features,
                 weight=w_class[y_int[tr]]),
            eval_set=Pool(train_df.iloc[va][ALL_FEATURES], y_int[va], cat_features=cat_features),
            use_best_model=True,
        )
        cb_oof[va] = model.predict_proba(train_df.iloc[va][ALL_FEATURES])
        cb_holdout += model.predict_proba(train_df.iloc[holdout_idx][ALL_FEATURES]) / N_SPLITS
        cb_test += model.predict_proba(test_df[ALL_FEATURES]) / N_SPLITS
        print(f"CatBoost fold {k + 1}: validation Macro F1 = {macro_f1(y_int[va], cb_oof[va].argmax(1)):.5f}")
        del model
        gc.collect()

    # Bobot blend dipilih HANYA dari OOF dev (holdout tetap bersih)
    blend_rows = []
    for w in np.round(np.arange(0, 1.01, 0.1), 1):
        p = w * lgb_result["oof"][dev_idx] + (1 - w) * cb_oof[dev_idx]
        blend_rows.append({"w_lgb": w, "dev_oof_macro_f1": macro_f1(y_int[dev_idx], p.argmax(1))})
    blend_table = pd.DataFrame(blend_rows)
    display(blend_table)

    w_best = float(blend_table.loc[blend_table["dev_oof_macro_f1"].idxmax(), "w_lgb"])
    final_dev_proba = w_best * lgb_result["oof"][dev_idx] + (1 - w_best) * cb_oof[dev_idx]
    final_holdout_proba = w_best * lgb_result["eval_pred"] + (1 - w_best) * cb_holdout
    final_test_proba = w_best * lgb_result["test_pred"] + (1 - w_best) * cb_test
    final_model_name = f"Blend LGB {w_best:.1f} / CatBoost {1 - w_best:.1f}"
    print(f"Ensemble: {final_model_name} | holdout Macro F1 (argmax) = "
          f"{macro_f1(y_int[holdout_idx], final_holdout_proba.argmax(1)):.5f}")
else:
    print("CatBoost dilewati (USE_CATBOOST = False).")

## 11.8 Optimasi multiplier per kelas untuk Macro F1

Multiplier dicari **hanya di OOF validation (dev)**, lalu diterapkan ke holdout dan test.
Dengan begitu skor holdout tetap merupakan estimasi yang jujur.

In [ ]:
def tune_multipliers(proba, y_true, candidates=None, max_rounds=4):
    if candidates is None:
        candidates = np.round(np.arange(0.5, 3.01, 0.1), 2)
    mult = np.ones(K)
    best = macro_f1(y_true, proba.argmax(1))
    history = [{"round": 0, "class": "-", "multiplier": 1.0, "macro_f1": best}]
    for r in range(max_rounds):
        improved = False
        for k in range(K):
            for c in candidates:
                trial = mult.copy()
                trial[k] = c
                s = macro_f1(y_true, (proba * trial).argmax(1))
                if s > best + 1e-6:
                    best, mult, improved = s, trial, True
            history.append({"round": r + 1, "class": class_labels[k],
                            "multiplier": mult[k], "macro_f1": best})
        if not improved:
            break
    return mult, best, pd.DataFrame(history)


class_multipliers, dev_tuned_f1, tuning_history = tune_multipliers(final_dev_proba, y_int[dev_idx])

display(pd.DataFrame({"class": class_labels, "multiplier": class_multipliers}))
print(f"Validation OOF Macro F1: argmax {macro_f1(y_int[dev_idx], final_dev_proba.argmax(1)):.5f} -> tuned {dev_tuned_f1:.5f}")

## 11.9 Evaluasi akhir — Macro F1 di Train / Validation / Test (holdout)

In [ ]:
dev_pred_argmax = final_dev_proba.argmax(1)
dev_pred_tuned = (final_dev_proba * class_multipliers).argmax(1)
holdout_pred_argmax = final_holdout_proba.argmax(1)
holdout_pred_tuned = (final_holdout_proba * class_multipliers).argmax(1)

# Sanity check: macro_f1 cepat == sklearn
assert abs(macro_f1(y_int[holdout_idx], holdout_pred_tuned)
           - f1_score(y_int[holdout_idx], holdout_pred_tuned, average="macro")) < 1e-9

evaluation_summary = pd.DataFrame([
    {"split": "Validation (OOF dev, 5-fold)", "postprocess": "argmax",
     "macro_f1": macro_f1(y_int[dev_idx], dev_pred_argmax),
     "accuracy": (dev_pred_argmax == y_int[dev_idx]).mean()},
    {"split": "Validation (OOF dev, 5-fold)", "postprocess": "multiplier (di-tune di sini, optimistis)",
     "macro_f1": macro_f1(y_int[dev_idx], dev_pred_tuned),
     "accuracy": (dev_pred_tuned == y_int[dev_idx]).mean()},
    {"split": "Test holdout (unseen)", "postprocess": "argmax",
     "macro_f1": macro_f1(y_int[holdout_idx], holdout_pred_argmax),
     "accuracy": (holdout_pred_argmax == y_int[holdout_idx]).mean()},
    {"split": "Test holdout (unseen)", "postprocess": "multiplier  <- METRIK UTAMA",
     "macro_f1": macro_f1(y_int[holdout_idx], holdout_pred_tuned),
     "accuracy": (holdout_pred_tuned == y_int[holdout_idx]).mean()},
])
evaluation_summary["model"] = final_model_name
display(evaluation_summary.style.format({"macro_f1": "{:.5f}", "accuracy": "{:.4%}"}))

holdout_report = pd.DataFrame(classification_report(
    y_int[holdout_idx], holdout_pred_tuned,
    labels=list(range(K)), target_names=class_labels,
    output_dict=True, zero_division=0,
)).T
print("Classification report — Test holdout (multiplier):")
display(holdout_report.style.format({"precision": "{:.4f}", "recall": "{:.4f}",
                                     "f1-score": "{:.4f}", "support": "{:.0f}"}))

cm = confusion_matrix(y_int[holdout_idx], holdout_pred_tuned, labels=list(range(K)))
plt.figure(figsize=(11, 9))
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels)
plt.title("Normalized Confusion Matrix — Test Holdout")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
plt.close()

In [ ]:
feature_importance = (
    lgb_result["importance"].groupby("feature")["gain"]
    .agg(["mean", "std"]).sort_values("mean", ascending=False).reset_index()
)
display(feature_importance.head(40))

top = feature_importance.head(30).iloc[::-1]
plt.figure(figsize=(10, 11))
plt.barh(top["feature"], top["mean"], xerr=top["std"].fillna(0))
plt.title("Top 30 Feature Importance (gain) — LightGBM")
plt.tight_layout()
plt.show()
plt.close()

## 11.10 (Opsional) Ablation per grup fitur

In [ ]:
if RUN_ABLATION:
    fast_params = {**lgb_params, "learning_rate": 0.1, "n_estimators": 2000}
    ablation_configs = {
        "E2: basis (skenario C+)": BASE_FEATURES,
        "E3: + target encoding": BASE_FEATURES + TE_FEATURES,
        "E4: + TE + freq/deviasi": ALL_FEATURES,
    }
    ablation_rows = []
    for name, feats in ablation_configs.items():
        print(f"\n=== {name} ({len(feats)} fitur) ===")
        res = run_lgb_cv(feats, dev_idx, dev_folds, holdout_idx, fast_params, log_period=0)
        mult, dev_tuned, _ = tune_multipliers(res["oof"][dev_idx], y_int[dev_idx])
        ablation_rows.append({
            "experiment": name,
            "n_features": len(feats),
            "val_oof_macro_f1": macro_f1(y_int[dev_idx], res["oof"][dev_idx].argmax(1)),
            "holdout_macro_f1_argmax": macro_f1(y_int[holdout_idx], res["eval_pred"].argmax(1)),
            "holdout_macro_f1_tuned": macro_f1(y_int[holdout_idx], (res["eval_pred"] * mult).argmax(1)),
        })
        del res
        gc.collect()
    ablation_table = pd.DataFrame(ablation_rows)
    display(ablation_table)
    ablation_table.to_csv(OUTPUT_DIR / "ablation.csv", index=False)
else:
    print("Ablation dilewati (RUN_ABLATION = False).")

## 11.11 (Opsional) Refit di 100% train untuk submission

Evaluasi di atas memakai 85% train. Jika `FINAL_REFIT_FULL_TRAIN = True`, TE dihitung ulang
dari seluruh train (OOF 5-fold) dan model dilatih ulang di 100% data. Multiplier kelas tetap
memakai hasil tuning dari 11.8. Tidak ada lagi holdout pada tahap ini, sehingga skor yang dilaporkan
tetap skor holdout di 11.9.

In [ ]:
if FINAL_REFIT_FULL_TRAIN:
    full_folds = np.full(n_train, -1, dtype=np.int8)
    for k, (_, va) in enumerate(skf.split(all_idx, y_int)):
        full_folds[va] = k

    attach_target_encoding(all_idx, full_folds, np.array([], dtype=np.int64))

    full_result = run_lgb_cv(ALL_FEATURES, all_idx, full_folds, np.array([], dtype=np.int64), lgb_params)
    display(full_result["fold_scores"])
    print(f"Full-train OOF Macro F1: {macro_f1(y_int, full_result['oof'].argmax(1)):.5f}")

    final_test_proba = full_result["test_pred"]
    final_model_name += " (refit 100% train)"
    if USE_CATBOOST:
        print("Catatan: refit hanya untuk LightGBM; blend CatBoost tidak diterapkan pada refit.")
else:
    print("Submission memakai rata-rata 5 model fold yang dilatih pada train (dev).")

## 11.12 Submission dan simpan artefak

In [ ]:
test_pred_int = (final_test_proba * class_multipliers).argmax(1)
test_predictions = np.array(class_labels)[test_pred_int]

sample_submission_path = DATA_DIR / "sample_submission.csv"
if sample_submission_path.exists():
    submission = pd.read_csv(sample_submission_path)
    prediction_map = pd.Series(test_predictions, index=test_ids)
    submission[TARGET] = submission[ID_COL].map(prediction_map)
    if submission[TARGET].isna().any():
        raise ValueError("Ada ID sample submission yang tidak punya prediksi.")
else:
    submission = pd.DataFrame({ID_COL: test_ids, TARGET: test_predictions})

assert len(submission) == n_test
assert submission[TARGET].isin(class_labels).all()

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

pred_dist = pd.DataFrame({
    "train": pd.Series(y_labels).value_counts(normalize=True),
    "test_prediction": pd.Series(test_predictions).value_counts(normalize=True),
}).reindex(class_labels).fillna(0)
display(pred_dist.style.format("{:.2%}"))

fold_scores.to_csv(OUTPUT_DIR / "fold_scores.csv", index=False)
evaluation_summary.to_csv(OUTPUT_DIR / "evaluation_summary.csv", index=False)
holdout_report.to_csv(OUTPUT_DIR / "holdout_classification_report.csv")
feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
tuning_history.to_csv(OUTPUT_DIR / "multiplier_tuning_history.csv", index=False)
np.save(OUTPUT_DIR / "test_proba.npy", final_test_proba)

with open(OUTPUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump({
        "model": final_model_name,
        "metric": "macro_f1",
        "classes": class_labels,
        "class_multipliers": [float(m) for m in class_multipliers],
        "class_weight_mode": CLASS_WEIGHT_MODE,
        "te_smoothing": TE_SMOOTHING,
        "n_features": len(ALL_FEATURES),
        "features": ALL_FEATURES,
        "split": {"train_dev": int(len(dev_idx)), "test_holdout": int(len(holdout_idx)),
                  "n_folds": N_SPLITS, "holdout_size": HOLDOUT_SIZE},
        "validation_oof_macro_f1": float(macro_f1(y_int[dev_idx], dev_pred_argmax)),
        "holdout_macro_f1_argmax": float(macro_f1(y_int[holdout_idx], holdout_pred_argmax)),
        "holdout_macro_f1_tuned": float(macro_f1(y_int[holdout_idx], holdout_pred_tuned)),
        "refit_full_train": FINAL_REFIT_FULL_TRAIN,
    }, f, ensure_ascii=False, indent=4)

print(f"Submission: {submission_path.resolve()} | shape {submission.shape}")
print(f"Artefak tersimpan di {OUTPUT_DIR.resolve()}")